In [1]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import tiktoken
from collections import defaultdict, Counter
import os
from dotenv import load_dotenv

/Users/pranitgunjal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# **Sentence Transformer**

In [2]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [3]:
train_df = pd.read_csv('../data/initial_datasets/dota2/dota2_train.csv')
test_df = pd.read_csv('../data/initial_datasets/dota2/dota2_test.csv')

In [4]:
train_df = train_df.sample(n=1000)

# **Tokenizer**

In [5]:
encoding = tiktoken.encoding_for_model("gpt-4")

In [6]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Get Co-Occurences**

In [10]:
train_df

,translated_message,label
2281,easy,0
1311,gg,0
184,"""boy churro the chaop""",0
2111,"""Yikes""",0
2311,XD,0
...,...,...
471,REPORT DAZZLE,0
1826,"""Really?>""",0
1403,"""friend, it turns out""",0
848,"""0 5""",0


In [11]:
text = train_df['translated_message'].to_list()

In [12]:
tokens_list = []
for sentence in text:
    token_ids = encoding.encode(sentence)
    # Optionally, get string versions of tokens
    tokens = [encoding.decode([tid]) for tid in token_ids]
    tokens_list.append(tokens)

In [13]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [14]:
top_k = 3

summary_text = ""
for token, counter in cooc.items():
    top = [w for w, _ in counter.most_common(top_k)]
    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"


In [15]:
print(summary_text)

'"' often appears with:  you, ?", ,.
'op' often appears with: ", ro,  ch.
'ro' often appears with: fl, ", op.
' ch' often appears with: ", op, ro.
' cha' often appears with: ", op, ro.
'boy' often appears with: ", op, ro.
' the' often appears with: ",  fuck, .".
'ur' often appears with: ", op, ro.
'Y' often appears with: ", WH,  ARE.
'ikes' often appears with: Y, ".
'it' often appears with: ", 's,  a.
''s' often appears with: ",  a, ,.
' a' often appears with: ",  to,  is.
' fact' often appears with: ", it, 's.
'?' often appears with: ", are,  you.
'ser' often appears with: ?, iously.
'iously' often appears with: ?, ser.
'(' often appears with: :'.
':'' often appears with: (.
'oki' often appears with: ",  ri, that.
' ri' often appears with: ", oki, that.
'that' often appears with: ", oki,  ri.
' pick' often appears with: ", oki,  ri.
'find' often appears with: ", vern,  Wy.
'vern' often appears with: find, ",  Wy.
' Wy' often appears with: find, ", vern.
'HA' often appears with: H, ", 

In [16]:
instruction = (
    "You are a data generator tasked with creating realistic DOTA 2 chat messages. "
    "These chat messages should be labeled according to their sentiment: toxic or non-toxic.\n"
    "Base the style on typical video game chat messages — include informal internet language, typos, and abbreviations\n"
    "You will be given statistics about the distribution, including average chat length, standard deviation, and most common words associated with each label and their frequency.\n"
    "Generate exactly 10 realistic DOTA 2 chat messages, one per line.\n"
    "Each line should follow this format: the chat message in double quotes, followed by a space and then the label (0 for toxic, 1 for non-toxic).\n"
    "No extra formatting — just plain text output, one line per comment.\n"
    "Here is the format:\n"
    "\"gg dawg\" 0\n"
    "\"I hate u bitch\" 1"
)
input = (
    f"Here are the token co-occurences ordered by frequency:\n{summary_text}",
    f"Now, generate the 10 new comments below:"
)

In [17]:
response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
print(response.output_text)

"ur such a joke, man" 0  
"nice play, well done!" 1  
"report this feeder pls" 0  
"let's go team, we got this!" 1  
"wtf were u doing??" 0  
"great teamwork guys" 1  
"stop feeding and play" 0  
"awesome ult there" 1  
"u guys r trash" 0  
"gl hf everyone!" 1


In [30]:
res = []
for i in tqdm(range(3)):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
    res.append(response.output_text)

100%|██████████| 3/3 [00:25<00:00,  8.51s/it]


In [31]:
labels = []
sentences = []
for i in range(3):
    for word in res[i].split("\n"):
        match = re.match(r'"(.*?)"\s*(-?\d+)', word)
        if match:
            quoted = match.group(1)      
            label = match.group(2)       
            sentences.append(quoted)
            labels.append(int(label))

In [35]:
generated_df = pd.DataFrame({
    'sentences': sentences,
    'labels': labels
})

In [32]:
second = generated_df

In [26]:
generated_df

,sentences,labels
0,"I'm done with this game, u guys are trash",0
1,"Nice job team, we got this",1
2,Why u feeding mid?,0
3,Let's push bot and end this!,1
4,Lol ur so bad at this,0
5,"Good luck, have fun everyone!",1
6,"Report this guy, so toxic",0
7,"Great teamwork, keep it up",1
8,Stop playing like noobs!,0
9,"Awesome save, well played",1


In [22]:
first = generated_df

In [36]:
combined = pd.concat([first, second, generated_df])

In [39]:
combined.to_csv('../data/generated/dota2/token_co_occurences/gen_token_co_occurences.csv', index=False)